In [ ]:
!python --version

In [ ]:
# Librerías
from google.colab import drive
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Montamos Google Drive en Colab
from google.colab import drive
drive.mount('/content/drive')

# Ruta base de trabajo
RUTA_BASE = "/content/drive/MyDrive/"

In [ ]:
# Importamos el dataset de Kaggle en formato csv
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

# Análisis exploratorio de datos (EDA)

In [ ]:
# Dimensiones del dataset
df.shape

In [ ]:
# Primeras observaciones
df.head()

In [ ]:
# Tipos de variables
df.info()

In [ ]:
# Comprobación de filas duplicadas
print("Filas duplicadas:", df.duplicated().sum())

## Valores faltantes

Cálculo del número de NaN por columna

In [ ]:
# Distribución inicial de smoking_status
df["smoking_status"].value_counts()

In [ ]:
# Consideramos los valores "Unknown" de smoking_status como NaN
df["smoking_status"] = df["smoking_status"].replace("Unknown", np.nan)

# Número y porcentaje de valores faltantes por variable
nan_por_columna = df.isna().sum().sort_values(ascending=False)
porcentaje_nan_por_columna = df.isna().mean().sort_values(ascending=False) * 100

# Resumen de valores faltantes
nan_columna_df = pd.DataFrame({
    'Num_NaN_Columna': nan_por_columna,
    'Porcentaje_Participantes': porcentaje_nan_por_columna.round(2)
    })
print(nan_columna_df)

In [ ]:
# Variables con presencia de NaN
nan_columna_df_plot = nan_columna_df[
    nan_columna_df['Num_NaN_Columna'] > 0]

# Visualización de valores faltantes
plt.figure(figsize=(10,5))
plt.bar(nan_columna_df_plot.index, nan_columna_df_plot['Porcentaje_Participantes'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('% de participantes con NaN')
plt.title('Valores faltantes por columna')
plt.tight_layout()
plt.show()

## Transformación de variables

In [ ]:
# Variables categóricas y numéricas
var_categoricas = ['gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'residence_type', 'smoking_status']
var_numericas = ['age', 'avg_glucose_level', 'bmi']

Transformamos las variables categóricas en factores

In [ ]:
# Género
df["gender"] = df["gender"].replace({
    "Male": "Hombre",
    "Female": "Mujer",
    "Other": np.nan
}).astype("category")

# Hipertensión
df["hypertension"] = df["hypertension"].map({
    0: "No",
    1: "Sí"
}).astype("category")

# Enfermedad cardiaca
df["heart_disease"] = df["heart_disease"].map({
    0: "No",
    1: "Sí"
}).astype("category")

# Estado civil
df["ever_married"] = df["ever_married"].map({
    "No": "No",
    "Yes": "Sí"
}).astype("category")

# Tipo de empleo
df["work_type"] = df["work_type"].replace({
    "children": "Niño",
    "Govt_job": "Empleado público",
    "Never_worked": "Nunca trabajó",
    "Private": "Sector privado",
    "Self-employed": "Autónomo"
}).astype("category")

# Tipo de residencia
df = df.rename(columns={"Residence_type": "residence_type"})
df["residence_type"] = df["residence_type"].map({
    "Rural": "Rural",
    "Urban": "Urbano"
}).astype("category")

# Tabaquismo
df["smoking_status"] = df["smoking_status"].replace({
    "formerly smoked": "Exfumador",
    "never smoked": "Nunca fumó",
    "smokes": "Fumador",
    "Unknown": np.nan
}).astype("category")

# Ictus (Variable objetivo)
df["stroke"] = df["stroke"].map({
    0: "No",
    1: "Sí"
}).astype("category")

## Estadísticas descriptivas

### Variable objetivo

In [ ]:
# Frecuencias de la variable Stroke
print(df['stroke'].value_counts())
print((df['stroke'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Distribución de la variable objetivo
plt.figure(figsize=(5,4))
df['stroke'].value_counts().plot(kind='bar')
plt.title("Distribución de Stroke")
plt.ylabel("Número de participantes")
plt.xticks(rotation=0)
plt.show()

### Variables numéricas

In [ ]:
# Estadísticas descriptivas
df[var_numericas].describe().round(2)

In [ ]:
# Histogramas
for col in var_numericas:
    plt.figure(figsize=(6,4))
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f'Distribución de {col}')
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.tight_layout()
    plt.show()

#### Detección de outliers

In [ ]:
# Boxplot variables numéricas
df[var_numericas].boxplot(figsize=(14,6), rot=45)
plt.title("Outliers en variables numéricas")
plt.show()

#### Correlaciones entre variables numéricas

In [ ]:
def matriz_correlacion(df, columnas, etiquetas=None, metodo="pearson",
                       anotar=True, figsize=(10, 9), enmascarar=True, nombre_archivo=None):
    corr = df[columnas].corr(method=metodo)
    if etiquetas:
        corr.index = etiquetas; corr.columns = etiquetas
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1) if enmascarar else None
    plt.figure(figsize=figsize)
    sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5, annot=anotar, fmt=".2f",
                annot_kws={"fontsize": 8},
                cbar_kws={"label": f"Coeficiente de correlación de {metodo.capitalize()}",
                          "shrink": 0.8})
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    if nombre_archivo:
        plt.savefig(RUTA_BASE + nombre_archivo, dpi=300, bbox_inches="tight")
    plt.show()

# Kaggle (3 variables, anotada y pequeña)
matriz_correlacion(df, ["age", "avg_glucose_level", "bmi"],
                   etiquetas=["Edad", "Glucosa media", "IMC"],
                   anotar=True, enmascarar=False, figsize=(4.5, 4),
                   nombre_archivo="figura_anexo_corr_kaggle.png")

### Variables categóricas

#### Count plot

In [ ]:
for col in var_categoricas:
    print(df[col].value_counts())

    fig, ax = plt.subplots(figsize=(6,4))
    sns.countplot(data=df, x=col, ax=ax)

    ax.set_title(f"Distribución de {col}")
    ax.grid(True)

    plt.show()

## Relación entre variables individuales y Stroke

### Variables numéricas vs Stroke

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(var_numericas) / n_cols))

plt.figure(figsize=(15, 3*n_rows))

for i, col in enumerate(var_numericas):
    plt.subplot(n_rows, n_cols, i+1)
    sns.boxplot(x='stroke', y=col, data=df)
    plt.title(col)
    plt.xlabel('')
    plt.ylabel('')

plt.tight_layout()
plt.show()

### Variables categóricas vs Stroke

In [ ]:
# Colores azul y naranja
colors = ['#ff7f0e', '#1f77b4']  # azul para Stroke Sí, naranja para Stroke No

for col in var_categoricas:
    if col != "stroke":
        # Tabla de porcentajes por fila (cada categoría suma 100%)
        tabla = pd.crosstab(df[col], df["stroke"], normalize='index') * 100
        tabla = tabla[['Sí', 'No']]  # Aseguramos el mismo orden de columnas
        print(f"\n{col} vs Stroke (%)")
        print(tabla.round(2))

        # Gráfico de barras apiladas usando la misma tabla
        ax = tabla.plot(kind='bar', stacked=True, figsize=(6,4), color=colors)
        plt.title(f"{col} vs Stroke (%)")
        plt.ylabel("Proporción de Stroke en cada grupo")
        plt.xlabel(col)
        plt.xticks(rotation=45)
        plt.ylim(0,100)
        plt.legend(title='Stroke')
        plt.tight_layout()
        plt.show()

In [ ]:
# Guardado y exportación del dataset final listo para el preprocesado
df_prep = df.copy()

drive.mount('/content/drive', force_remount=True)
df_prep.to_csv("/content/drive/MyDrive/df_prep.csv", index=False)